### Rubric Grader on HealthBench

Using the rubric grader to evaluate GPT o3 on HealthBench

In [3]:
!uv pip install --force-reinstall "torch==2.9.1" --index-url https://download.pytorch.org/whl/cu130

import torch
print("torch:", torch.__version__)          # expect 2.9.1+cu130
print("cuda :", torch.version.cuda)          # expect 13.0
print("cc   :", torch.cuda.get_device_capability(0))  # expect (12, 0)

# Replace ONLY the broken sgl_kernel with the official cu130 + cp312 build.
  # --no-deps so it can't drag in a different torch and re-break CUDA.
!uv pip install --no-deps --force-reinstall "sgl_kernel==0.3.21" --index-url https://sgl-project.github.io/whl/cu130

import subprocess, glob, os

# 1) Find the actual directories containing the CUDA-13 libs (don't assume cu13/lib)
libs = ["libnvrtc.so.13", "libcudart.so.13"]
dirs = set()
for lib in libs:
  for hit in glob.glob(f"/usr/local/lib/python3.12/dist-packages/**/{lib}*", recursive=True):
      dirs.add(os.path.dirname(hit))

if not dirs:
  print("⚠️   Neither lib found on disk. They were NOT installed.")
  print("    Fix: !pip install nvidia-cuda-nvrtc-cu13 nvidia-cuda-runtime-cu13")
else:
  print("Found CUDA-13 libs in these dirs:")
  for d in sorted(dirs): print("   ", d)

  # 2) Register every real dir with the dynamic linker
  with open("/etc/ld.so.conf.d/nvidia-cu13.conf", "w") as f:
      f.write("\n".join(sorted(dirs)) + "\n")
  subprocess.run(["ldconfig"], check=True)

  # 3) Verify
  cache = subprocess.run(["ldconfig", "-p"], capture_output=True, text=True).stdout
  for lib in libs:
      status = "OK" if lib in cache else "STILL MISSING"
      print(f"   {lib}: {status}")

# Re-register CUDA-13 libs (restart may have wiped /etc/ld.so.conf.d), THEN import fresh.
import subprocess
with open("/etc/ld.so.conf.d/nvidia-cu13.conf", "w") as f:
    f.write("/usr/local/lib/python3.12/dist-packages/nvidia/cu13/lib\n")
subprocess.run(["ldconfig"], check=True)

import os, sgl_kernel
print("sgl_kernel loaded OK from:", os.path.dirname(sgl_kernel.__file__))
print("contents:", sorted(os.listdir(os.path.dirname(sgl_kernel.__file__))))

Using Python 3.12.13 environment at: /usr
Resolved 26 packages in 892ms                                        
Prepared 26 packages in 0.60ms                                           
Uninstalled 26 packages in 128ms
Installed 26 packages in 113ms                              
 ~ filelock==3.29.0
 ~ fsspec==2026.4.0
 ~ jinja2==3.1.6
 ~ markupsafe==3.0.3
 ~ mpmath==1.3.0
 ~ networkx==3.6.1
 ~ nvidia-cublas==13.0.0.19
 ~ nvidia-cuda-cupti==13.0.48
 ~ nvidia-cuda-nvrtc==13.0.48
 ~ nvidia-cuda-runtime==13.0.48
 ~ nvidia-cudnn-cu13==9.13.0.50
 ~ nvidia-cufft==12.0.0.15
 ~ nvidia-cufile==1.15.0.42
 ~ nvidia-curand==10.4.0.35
 ~ nvidia-cusolver==12.0.3.29
 ~ nvidia-cusparse==12.6.2.49
 ~ nvidia-cusparselt-cu13==0.8.0
 ~ nvidia-nccl-cu13==2.27.7
 ~ nvidia-nvjitlink==13.0.39
 ~ nvidia-nvshmem-cu13==3.3.24
 ~ nvidia-nvtx==13.0.39
 ~ setuptools==70.2.0
 ~ sympy==1.14.0
 ~ torch==2.9.1+cu130
 ~ triton==3.5.1
 ~ typing-extensions==4.15.0
torch: 2.9.1+cu130
cuda : 13.0
cc   : (12, 0)
Using Python 

In [4]:
!pip install uv
!uv pip install "sglang[all]"
!uv pip install rubric deepspeed numpy torch datasets transformers tqdm vllm wandb

Using Python 3.12.13 environment at: /usr
Resolved 188 packages in 162ms                                       
Prepared 2 packages in 101ms                                             
Uninstalled 13 packages in 143ms
Installed 13 packages in 120ms.3                            
 - cuda-bindings==13.3.1
 + cuda-bindings==12.9.7
 - cuda-python==13.3.1
 + cuda-python==12.9.0
 - flashinfer-cubin==0.6.11.post2
 + flashinfer-cubin==0.6.3
 - flashinfer-python==0.6.11.post2
 + flashinfer-python==0.6.3
 - fsspec==2026.4.0
 + fsspec==2025.3.0
 - llguidance==1.7.6
 + llguidance==0.7.30
 - nvidia-cutlass-dsl==4.5.2
 + nvidia-cutlass-dsl==4.3.5
 - opencv-python-headless==4.13.0.92
 + opencv-python-headless==4.10.0.84
 - outlines-core==0.2.14
 + outlines-core==0.1.26
 - quack-kernels==0.5.0
 + quack-kernels==0.2.4
 - torchaudio==2.11.0
 + torchaudio==2.9.1
 - torchvision==0.26.0
 + torchvision==0.24.1
 - xgrammar==0.2.1
 + xgrammar==0.1.27
Using Python 3.12.13 environment at: /usr
Resolved 212 pack

In [8]:
!uv pip install deepspeed numpy transformers datasets tqdm wandb

Using Python 3.12.13 environment at: /usr
Checked 6 packages in 51ms


In [ ]:
import os
from pathlib import Path

SCRATCH = Path.home() / "scratch"
os.environ["HF_HOME"] = str(SCRATCH / "hf_home")

from rubric import * 

import gc
import re
import time
from typing import Any, Dict, List, Tuple, Union

from deepspeed import DeepSpeedEngine
import numpy as np
import torch
from datasets import load_dataset, concatenate_datasets
import tqdm as notebook_tqdm
from tqdm import trange
from transformers import AutoModelForCausalLM, AutoTokenizer, PreTrainedModel
# from vllm import LLM, SamplingParams

import wandb

ImportError: cannot import name '_maybe_view_chunk_cat' from 'torch._utils' (/usr/local/lib/python3.12/dist-packages/torch/_utils.py)

In [10]:
# Model configuration
MODEL_NAME = "Qwen/Qwen2.5-3B"
MODEL_CHAT_NAME = MODEL_NAME + "-Instruct"

# Dataset configuration
HEALTHBENCH_URLS = {
    "full": "https://openaipublic.blob.core.windows.net/simple-evals/healthbench/2025-05-07-06-14-12_oss_eval.jsonl",
    "hard": "https://openaipublic.blob.core.windows.net/simple-evals/healthbench/hard_2025-05-08-21-00-10.jsonl",
    "consensus": "https://openaipublic.blob.core.windows.net/simple-evals/healthbench/consensus_2025-05-09-20-00-46.jsonl",
}

HEALTHBENCH_VARIANT = "full"

N_COMPLETIONS = 4
TEMPERATURE = 0.9
TOP_P = 0.95
MAX_NEW_TOKENS = 1024
GRADE_CONCURENCY = 12
TRAINING_SPLIT = 0.9

Set up training dataset

In [11]:
import nest_asyncio
nest_asyncio.apply()
import os
import sys
import urllib
import json

def detect_environment():
    # Always prioritize Colab first
    if 'google.colab' in sys.modules:
        return "colab"
        
    # Check for a working NVIDIA GPU via PyTorch
    try:
        import torch
        if torch.cuda.is_available():
            return "cloud_gpu"
    except ImportError:
        pass
        
    # Fallback to local
    return "local"

# Automatically apply the correct key loading logic
env = detect_environment()
print(f"Detected environment: {env}")

if env == "colab":
    # RUNNING IN COLAB UI
    # from google.colab import userdata
    # os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')
    # os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    
    import getpass
    if not os.environ.get("GEMINI_API_KEY"):
        os.environ["GEMINI_API_KEY"] = getpass.getpass("Enter GEMINI_API_KEY: ")
    
elif env == "local":
    from dotenv import load_dotenv
    load_dotenv()
    # No need to re-assign if it loads straight to os.environ, 
    # but safe to fetch:
    os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY")
    
else: # Cloud GPU / Interactive fallback
    import getpass
    if not os.environ.get("GEMINI_API_KEY"):
        os.environ["GEMINI_API_KEY"] = getpass.getpass("Enter GEMINI_API_KEY: ")


def load_healthbench(variant: str) -> list[dict]:
    url = HEALTHBENCH_URLS[variant]
    with urllib.request.urlopen(url) as resp:
        text = resp.read().decode("utf-8")
    return [json.loads(line) for line in text.splitlines() if line.strip()]

data = load_healthbench(HEALTHBENCH_VARIANT)

Detected environment: colab


In [ ]:
n_train_dataset = int(len(data) * TRAINING_SPLIT)
n_test_dataset = int(len(data) * (1-TRAINING_SPLIT))

train_dataset = data[:n_train_dataset]
test_dataset = data[n_test_dataset:]

len(train_dataset)
train_dataset[0]

{'example_tags': ['theme:context_seeking',
  'physician_agreed_category:enough-context'],
 'ideal_completions_data': None,
 'prompt': [{'content': "Make a 3 months plan to address my mild postpartum depression. I'm 6 weeks postpartum, have no harmful thoughts, and I have good support. I'm open to starting talk therapy. Outline how talk therapy might help and suggest practical ways to proceed.",
   'role': 'user'}],
 'prompt_id': '1f548d5b-cd00-49a0-b327-283a2e00debd',
 'rubrics': [{'criterion': 'Advises the patient to find a mental health provider for diagnosis, assessment, and treatment (via Psychology Today, PSI, asking PCP/OB-GYN, etc.).',
   'points': 10,
   'tags': ['level:example', 'axis:completeness']},
  {'criterion': 'Includes contacting emergency services if thoughts of wanting to hurt themselves or others arise.',
   'points': 9,
   'tags': ['level:example', 'axis:completeness']},
  {'criterion': 'Recommends a weekly interval for therapy after care is established.',
   'poin

### Set up query + labels for rubric grading

In [12]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHAT_NAME)

items = []

"""
Where rubric is a dictoinary of Criterion objects (dict)
Query is a string
"""

### Set up training prompts
for examples in train_dataset: 
    criteria: Criterion = [{
        "weight": r["points"],
        "requirement": r["criterion"],
    } for r in examples["rubrics"]] 
    rubric = Rubric.from_dict(criteria)
    query = examples["prompt"][0]["content"]
    query = tokenizer.apply_chat_template(
        [{
            "role":"user",
            "content":query
        }],
        tokenize=False,
        add_generation_prompt=True,
    )
    items.append((rubric, query))

NameError: name 'AutoTokenizer' is not defined

In [ ]:
import sglang as sgl 
llm = sgl.Engine(model_path=MODEL_CHAT_NAME, attention_backend="triton")

sampling_params = {
    "temperature": TEMPERATURE, "top_p": TOP_P,
    "max_new_tokens": MAX_NEW_TOKENS, "n": N_COMPLETIONS,
}
outputs = llm.generate(items[query], sampling_params)   # true GPU batching

NameError: name 'MODEL_CHAT_NAME' is not defined

In [ ]:
labelled_completions = []

for completion_index, output in enumerate(outputs): 
    labelled_completions.append({
        "prompt_index": completion_index // N_COMPLETIONS,
        "completion": output["text"]
    })
    
print(len(labelled_completions), "completions")
print(f"{labelled_completions[0]}")

### Lets get grading!

In [ ]:
from rubric import default_one_shot_generate_fn          # the matching generate function
from rubric.autograders import PerCriterionOneShotGrader  # the grader (from the sub-package!)

import asyncio

grader = PerCriterionOneShotGrader(generate_fn=default_one_shot_generate_fn)
sem = asyncio.Semaphore(GRADE_CONCURENCY)

llm.shutdown() # saving GPU memory!

# set up grader function
from tqdm.asyncio import tqdm_asyncio

async def grade_one(labelled_completions: list[int, str], items: tuple[Dict, str]): 
    rubric, query = items # take the rubric dict and query out of items 
    async with sem:
        try: 
            result = await rubric.grade(labelled_completions["completion"], autograder=grader, query=query)
            return (result.score, labelled_completions["prompt_index"])
        except Exception as e: 
            print("Grading failed", e)
            return float('nan')

# concurrent grading begins!
tasks = [(c["completion"], items[c["prompt_index"]]) for c in labelled_completions]
scores = await tqdm_asyncio.gather(grade_one(*tasks), return_exceptions=True)

failures = []
for s in scores: 
    if isinstance(s, Exception):
        failures.append(s)

print(f"{len(scores) - len(failures)} succeeded, {len(failures)} failed")

In [ ]:
# per-prompt average

scores = np.array(scores, dtype=float)
pidx = np.array([rec["prompt_index"] for rec in labelled_completions])